In [1]:
import torch
import whisper
import sounddevice as sd
import soundfile as sf
import numpy as np
from pathlib import Path
from datetime import datetime

class AudioTranscriber:
    def __init__(self, model_size="base"):
        """
        Initialize the transcriber with specified Whisper model size.
        Available sizes: tiny, base, small, medium, large
        """
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Using device: {self.device}")
        self.model = whisper.load_model(model_size).to(self.device)
        
        # Recording parameters
        self.sample_rate = 16000
        self.channels = 1
    
    def record_audio(self, duration=10, filename=None):
        """
        Record audio for specified duration and save to file.
        Args:
            duration (int): Recording duration in seconds
            filename (str): Optional filename to save the recording
        Returns:
            str: Path to the saved audio file
        """
        print(f"Recording for {duration} seconds...")
        
        # Record audio
        recording = sd.rec(
            int(duration * self.sample_rate),
            samplerate=self.sample_rate,
            channels=self.channels
        )
        sd.wait()
        
        # Generate filename if not provided
        if filename is None:
            filename = f"recording_{datetime.now().strftime('%Y%m%d_%H%M%S')}.wav"
        
        # Create recordings directory if it doesn't exist
        Path("recordings").mkdir(exist_ok=True)
        filepath = Path("recordings") / filename
        
        # Save the recording
        sf.write(filepath, recording, self.sample_rate)
        print(f"Recording saved to {filepath}")
        return str(filepath)
    
    def transcribe_file(self, audio_path):
        """
        Transcribe an audio file to text.
        Args:
            audio_path (str): Path to the audio file
        Returns:
            dict: Transcription results including text and segments
        """
        print(f"Transcribing {audio_path}...")
        try:
            result = self.model.transcribe(audio_path)
            return result
        except Exception as e:
            print(f"Error during transcription: {str(e)}")
            return None
    
    def transcribe_live(self, duration=10):
        """
        Record and transcribe audio in one go.
        Args:
            duration (int): Recording duration in seconds
        Returns:
            dict: Transcription results
        """
        audio_path = self.record_audio(duration)
        return self.transcribe_file(audio_path)

def main():
    # Example usage
    transcriber = AudioTranscriber(model_size="base")
    
    # Option 1: Transcribe existing audio file
    result = transcriber.transcribe_file("recording.wav")
    if result:
        print("\nTranscription:")
        print(result["text"])
        
        print("\nDetailed segments:")
        for segment in result["segments"]:
            print(f"[{segment['start']:.1f}s -> {segment['end']:.1f}s] {segment['text']}")
    
    # Option 2: Record and transcribe live audio
    print("\nRecording and transcribing live audio...")
    result = transcriber.transcribe_live(duration=10)
    if result:
        print("\nTranscription:")
        print(result["text"])

if __name__ == "__main__":
    main()

/home/beybars/Desktop/beybars/projects/side_hustle/EKTU/venv/lib/python3.12/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at ../c10/cuda/CUDAFunctions.cpp:108.)
  return torch._C._cuda_getDeviceCount() > 0
/home/beybars/Desktop/beybars/projects/side_hustle/EKTU/venv/lib/python3.12/site-packages/whisper/__init__.py:150: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limit

Using device: cpu
Transcribing recording.wav...


/home/beybars/Desktop/beybars/projects/side_hustle/EKTU/venv/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Transcription:
 Hello. Hi there. This is Jamie from STL Truckers LLC. Am I speaking with Samuel Kerney? Yeah. Great. I see that you previously contacted our HR agency about the company driver position. Are you still open to this agency? Yeah, what company is this? We're STL Truckers LLC based in St. Louis, Missouri. We operate across all 48 states and are currently looking for company drivers. Would you like me to give you a brief overview of the position? Yes. The company driver position offers a starting pay of $62.63 per mile depending on your background and experience. After four or six weeks, there's potential for an increase to $0.65 per mile based on your performance. We have a structured home time policy and our fleet consists of modern trucks equipped with amenities for your comfort. Our routes cover all 48 states. So flexibility and destinations is essential. Can you tell me about your experience with OTR driving and are there any geographic limitations that would prevent yo

/home/beybars/Desktop/beybars/projects/side_hustle/EKTU/venv/lib/python3.12/site-packages/whisper/transcribe.py:126: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")



Transcription:
 1.5% 1.5% 1.5% 1.5% 1.5% 1.5% 1.5%
